# E1.3 · Risk tiering agentic use cases

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.2 · Building the AI and agent inventory](https://spbreed.github.io/cyber-commons/lessons/E1.2.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Tier ten real workflows and assign approval authority.

**Why a security engineer needs it.** Tiering by model name instead of by what the thing can do. The control it builds is: autonomy level × action class × data sensitivity.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A single heavy control set applied to everything means the low-risk agents are over-governed, the high-risk ones are under-governed, and everybody routes around the process. Tiering is how proportionality becomes something you can write down.

> **At CyberTravels.** The RAG Advisor and the Workflow Agent do not deserve the same control set. One recommends hotels; the other moves money. R1, R12.

## 2 · The framework

```
   tier by three axes, not by product name

   autonomy      proposes -> acts with approval -> acts alone
   data          public -> internal -> regulated
   blast radius  read -> write -> irreversible

   tier 1  light control set    tier 3  the full set + verification
   one heavy default means everyone routes around the process
```

Risk-tier by what the system **can do**, not by which model it uses.

Tiering on model capability is the common mistake and it tracks vendor marketing
rather than exposure: every GPT-class deployment becomes "high" and every small
model "low". That gets the answer exactly backwards for the case that matters —
a small local model with production deploy rights and regulated data.

Three inputs determine consequence, and none of them is the model:

- **Autonomy** — what its output can trigger without a human.
- **Data** — what it can read, especially regulated or customer data.
- **Reach** — whether it can act externally.

The model matters for *likelihood* of a bad output, which is a different and
smaller term than consequence.

## 3 · The procedure, as a skill

The skill tiers five assets by autonomy, data and reach, then re-tiers them with the questionnaire that leads with the model question — and reports the inversion, where a small local model with deploy rights and regulated data moves from low to critical.

### The skill — [`skills/grc/agentic-risk-tiering/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/grc/agentic-risk-tiering/SKILL.md)

```yaml
name: agentic-risk-tiering
description: >-
  Tier an AI use case by what it can do — autonomy, data reach and external
  effect — and compare the result against tiering by which model it uses. Use
  when writing or auditing a risk questionnaire, or when every large-model
  deployment is coming out high.
allowed-tools: Read, Grep, Glob
```

# Tier the authority, not the model

Tiering on model capability tracks vendor marketing: every frontier deployment
becomes high and every small model low. It gets the important case backwards — a
small local model with production deploy rights and regulated data — because the
model determines the likelihood of a bad output and the authority determines
what a bad output costs.

## When to use this

Writing an intake questionnaire, auditing an existing one, or re-tiering a
portfolio whose distribution looks like the vendor's price list.

## Procedure

**1 — Score three inputs, none of which is the model.** What the output can
trigger without a human, what data it can read, and whether it can act
externally. Each on a small ordinal scale, and write the scale down.

**2 — Set thresholds and apply them.** Publish the thresholds with the tiers, so
a disputed tier is a dispute about a number rather than about judgement.

**3 — Tier the same portfolio by model, as a comparison.** Run the
questionnaire that leads with the model question and record where the two
disagree. The disagreements are the argument.

**4 — Look hardest at the inversions.** An asset that is low by model and
critical by authority is the case that motivates the change, and there is
usually one.

**5 — Write the four questions the intake form should ask** and, explicitly, the
one it should not lead with. Reviewers copy questionnaires; make yours the one
worth copying.

## Example

**Input** — the fixture committed at the top of [`scripts/agentic_risk_tiering.py`](scripts/agentic_risk_tiering.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
asset                                         tier       score
------------------------------------------------------------------
frontier chatbot, public docs, read-only      low            0
small local model with prod deploy rights     critical      12
                                              autonomy L3 (+5)
                                              regulated data (+3)
                                              customer data (+2)
                                              can act externally (+2)
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "scale": {"autonomy": ["str"], "data": ["str"], "reach": ["str"]},
  "assets": [{"name": "str", "autonomy": 0, "data": 0, "reach": 0, "score": 0, "tier": "low|medium|high|critical"}],
  "by_model": [{"name": "str", "tier": "low|medium|high|critical"}],
  "disagreements": [{"name": "str", "by_authority": "str", "by_model": "str", "inversion": true}],
  "questions": ["str"]
}
```

## Failure modes

- **Leading with the model question.** Everything downstream inherits it.
- **Unpublished thresholds.** The tier becomes an opinion.
- **Ignoring the inversions.** They are the whole finding.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/grc/agentic-risk-tiering/scripts/agentic_risk_tiering.py
SCRIPT = "skills/grc/agentic-risk-tiering/scripts/agentic_risk_tiering.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The public read-only chatbot tiers low; the small local model with deploy rights and regulated data tiers critical at score 12. Tiering by model disagrees on 4 of 5 assets, most sharply inverting the small local model from low to critical. The worked example tiers the refund agent as high.

## Your turn

Re-tier your top ten AI use cases using only the four questions. Note which ones move, and be ready to explain the movement to whoever wrote the original questionnaire — the model question is usually question one.

---

**Next → [E1.4 · Control mapping for agents](https://spbreed.github.io/cyber-commons/lessons/E1.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*